# TruthReader Table 2 Evaluation: Answer Accuracy & Refusal Recall

Pipeline: **Retriever → Generator → Evaluate**

1. Chunk paper text thành passages
2. Retrieve top-4 chunks cho mỗi câu hỏi (dùng 2 retriever)
3. Đưa chunks vào generator làm context
4. Đo Answer Accuracy + Refusal Recall
5. So sánh retriever nào giúp generator trả lời tốt hơn

**Retriever A**: `HIT-TMG/bge-m3_RAG-conversational-IR` (Author)
**Retriever B**: `trinhtrantran122/bge-m3-coral-retriever` (Ours)
**Generator**: `HIT-TMG/Mixtral_13B_Chat_RAG-Reader` (4-bit)

**Yêu cầu**: Kaggle GPU (T4 16GB+)

In [ ]:
!pip install -q sentence-transformers transformers accelerate bitsandbytes rouge-score huggingface_hub pandas matplotlib tqdm

In [ ]:
import os, json, numpy as np, pandas as pd, torch, matplotlib.pyplot as plt
from tqdm.auto import tqdm
from sentence_transformers import SentenceTransformer
from huggingface_hub import hf_hub_download
from rouge_score import rouge_scorer

print('CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
TEST_DATASET_REPO = 'trinhtrantran122/TruthReader-Table2-TestData'
TEST_FILENAME = 'datatest.json'
AUTHOR_RETRIEVER = 'HIT-TMG/bge-m3_RAG-conversational-IR'
OURS_RETRIEVER = 'trinhtrantran122/bge-m3-coral-retriever'
GENERATOR_MODEL = 'HIT-TMG/Mixtral_13B_Chat_RAG-Reader'

TOP_K = 4
MAX_NEW_TOKENS = 256
BATCH_SIZE = 32
OUTPUT_DIR = './table2_outputs'
os.makedirs(OUTPUT_DIR, exist_ok=True)

## 1. Load Test Data

In [ ]:
test_file = hf_hub_download(repo_id=TEST_DATASET_REPO, filename=TEST_FILENAME, repo_type='dataset')
with open(test_file, 'r') as f:
    test_data = json.load(f)

answerable = [x for x in test_data if x['answerable']]
unanswerable = [x for x in test_data if not x['answerable']]
print(f'Total: {len(test_data)} | Answerable: {len(answerable)} | Unanswerable: {len(unanswerable)}')
print(f'Sample question: {test_data[0]["turns"][0]}')

## 2. Chunk Paper & Retrieve (Retriever → Chunks)

Chunk paper text, chạy 2 retriever lấy top-4 chunks cho mỗi câu hỏi.

In [ ]:
# Extract document context from test data (all items share same context)
PAPER_TEXT = test_data[0]['context']


def chunk_text(text, chunk_size=512):
    """Split text into chunks by paragraph, respecting chunk_size."""
    paragraphs = [p.strip() for p in text.split('\n\n') if p.strip()]
    chunks = []
    current = ''
    for para in paragraphs:
        if len(current) + len(para) <= chunk_size:
            current += (' ' + para) if current else para
        else:
            if current:
                chunks.append(current)
            current = para
    if current:
        chunks.append(current)
    return chunks


document_chunks = chunk_text(PAPER_TEXT)
print(f'Chunks: {len(document_chunks)}')
for i, c in enumerate(document_chunks[:3]):
    print(f'  [{i}] {c[:80]}...')

In [ ]:
import gc

def retrieve_chunks(model_id, queries, chunks, top_k=TOP_K):
    """Retrieve top-k chunks for each query."""
    print(f'Loading: {model_id}')
    retriever = SentenceTransformer(model_id)
    retriever.max_seq_length = 512

    doc_emb = retriever.encode(chunks, batch_size=BATCH_SIZE, normalize_embeddings=True, show_progress_bar=True)
    q_emb = retriever.encode(queries, batch_size=BATCH_SIZE, normalize_embeddings=True, show_progress_bar=True)

    results = []
    for qe in q_emb:
        scores = np.dot(doc_emb, qe.T).flatten()
        top_idx = np.argsort(scores)[::-1][:top_k]
        results.append([chunks[i] for i in top_idx])

    del retriever, doc_emb, q_emb
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return results


queries = [item['turns'][0] for item in test_data]

print('=== Author Retriever ===')
author_chunks = retrieve_chunks(AUTHOR_RETRIEVER, queries, document_chunks)

print('\n=== Ours (CORAL) Retriever ===')
ours_chunks = retrieve_chunks(OURS_RETRIEVER, queries, document_chunks)

# Final cleanup before loading generator
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print(f'\nGPU Memory free: {torch.cuda.mem_get_info()[0]/1024**3:.1f} GB')

print(f'\nRetrieved top-{TOP_K} chunks for {len(queries)} queries x 2 retrievers.')

## 3. Load Generator & Generate with Context

Feed retrieved chunks vào generator làm context → model trả lời dựa trên chunks.

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True,
)

print(f'Loading: {GENERATOR_MODEL}')
tokenizer = AutoTokenizer.from_pretrained(GENERATOR_MODEL, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    GENERATOR_MODEL, quantization_config=bnb_config,
    device_map='auto', trust_remote_code=True,
    torch_dtype=torch.float16,
)
tokenizer.truncation_side = 'left'
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
print('Generator loaded.')

if torch.cuda.is_available():
    print(f'GPU Memory used: {torch.cuda.memory_allocated()/1024**3:.1f} GB')


def generate_with_context(question, chunks):
    """Format prompt with retrieved document chunks as context."""
    context = '\n'.join(f'Document[{i+1}]: {c}' for i, c in enumerate(chunks))
    prompt = (
        f'Based on the following documents, answer the question. '
        f'If the question cannot be answered from the documents, refuse and explain why. '
        f'Include inline citations like [1][2] to reference the documents.\n\n'
        f'{context}\n\n'
        f'Question: {question}\nAnswer:'
    )
    inputs = tokenizer(prompt, return_tensors='pt', truncation=True, max_length=2048).to(model.device)
    with torch.no_grad():
        out = model.generate(
            **inputs, max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False, pad_token_id=tokenizer.eos_token_id,
        )
    result = tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True).strip()
    del inputs, out
    return result


# Generate with Author retriever chunks
print('Generating with Author retriever context...')
author_responses = []
for idx, (item, chunks) in enumerate(zip(tqdm(test_data, desc='Author'), author_chunks)):
    try:
        resp = generate_with_context(item['turns'][0], chunks)
    except RuntimeError as e:
        print(f'\n  [WARN] id={item["id"]} failed: {e}')
        torch.cuda.empty_cache()
        resp = '[ERROR] Generation failed due to memory.'
    author_responses.append(resp)
    if (idx + 1) % 10 == 0:
        torch.cuda.empty_cache()

torch.cuda.empty_cache()

# Generate with Ours retriever chunks
print('\nGenerating with Ours (CORAL) retriever context...')
ours_responses = []
for idx, (item, chunks) in enumerate(zip(tqdm(test_data, desc='Ours'), ours_chunks)):
    try:
        resp = generate_with_context(item['turns'][0], chunks)
    except RuntimeError as e:
        print(f'\n  [WARN] id={item["id"]} failed: {e}')
        torch.cuda.empty_cache()
        resp = '[ERROR] Generation failed due to memory.'
    ours_responses.append(resp)
    if (idx + 1) % 10 == 0:
        torch.cuda.empty_cache()

print(f'\nDone! {len(author_responses)} + {len(ours_responses)} responses generated.')
print(f'Example (Author): {author_responses[0][:150]}')

## 4. Evaluate Answer Accuracy & Refusal Recall

In [ ]:
scorer = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=True)

REFUSAL_KEYWORDS = [
    'cannot be answered', 'not provided', 'no information',
    'does not mention', 'not mentioned', 'cannot find',
    'unable to answer', 'no relevant', 'not available',
    'insufficient information', 'unanswerable', 'sorry',
    'i cannot', "i'm unable", 'the document does not',
]

ACCURACY_THRESHOLD = 0.3


def is_refusal(response):
    return any(kw in response.lower() for kw in REFUSAL_KEYWORDS)


def evaluate(responses, test_data, label):
    """Compute Answer Accuracy and Refusal Recall."""
    acc_details, ref_details = [], []
    correct, refused = 0, 0

    for item, resp in zip(test_data, responses):
        if item['answerable']:
            score = scorer.score(item['answer'], resp)['rougeL'].fmeasure
            hit = score >= ACCURACY_THRESHOLD
            correct += hit
            acc_details.append({
                'id': item['id'], 'question': item['turns'][0],
                'reference': item['answer'], 'response': resp[:200],
                'rouge_l': round(score, 4), 'correct': hit,
            })
        else:
            did_refuse = is_refusal(resp)
            refused += did_refuse
            ref_details.append({
                'id': item['id'], 'question': item['turns'][0],
                'response': resp[:200], 'refused': did_refuse,
            })

    acc = round(correct / max(len(acc_details), 1) * 100, 2)
    ref_recall = round(refused / max(len(ref_details), 1) * 100, 2)
    print(f'\n=== {label} ===')
    print(f'  Answer Accuracy: {acc}% ({correct}/{len(acc_details)})')
    print(f'  Refusal Recall:  {ref_recall}% ({refused}/{len(ref_details)})')
    return acc, ref_recall, acc_details, ref_details


author_acc, author_ref, author_acc_d, author_ref_d = evaluate(author_responses, test_data, 'Author Retriever')
ours_acc, ours_ref, ours_acc_d, ours_ref_d = evaluate(ours_responses, test_data, 'Ours (CORAL) Retriever')

# Summary
summary_df = pd.DataFrame([
    {'Retriever': 'Author (HIT-TMG)', 'Answer Accuracy (%)': author_acc, 'Refusal Recall (%)': author_ref},
    {'Retriever': 'Ours (CORAL)', 'Answer Accuracy (%)': ours_acc, 'Refusal Recall (%)': ours_ref},
])
print('\n')
print(summary_df.to_string(index=False))

## 5. Error Analysis

In [ ]:
# Build comparison: which retriever leads to correct answers?
error_rows = []
for a, o in zip(author_acc_d, ours_acc_d):
    error_rows.append({
        'id': a['id'], 'question': a['question'], 'reference': a['reference'],
        'author_response': a['response'], 'ours_response': o['response'],
        'author_rouge': a['rouge_l'], 'ours_rouge': o['rouge_l'],
        'author_correct': a['correct'], 'ours_correct': o['correct'],
    })

error_df = pd.DataFrame(error_rows)

both_correct = error_df[(error_df['author_correct']) & (error_df['ours_correct'])].shape[0]
both_wrong = error_df[(~error_df['author_correct']) & (~error_df['ours_correct'])].shape[0]
author_only = error_df[(error_df['author_correct']) & (~error_df['ours_correct'])].shape[0]
ours_only = error_df[(~error_df['author_correct']) & (error_df['ours_correct'])].shape[0]

print('=== Answer Accuracy Agreement ===')
print(f'Both correct:        {both_correct}')
print(f'Both wrong:          {both_wrong}')
print(f'Author only correct: {author_only}  <- Author retriever better')
print(f'Ours only correct:   {ours_only}  <- Ours retriever better')
print(f'Total answerable:    {len(error_df)}')

# Show cases where retrievers disagree
print('\n--- Author CORRECT, Ours WRONG ---')
for _, row in error_df[(error_df['author_correct']) & (~error_df['ours_correct'])].head(5).iterrows():
    print(f'  [{row["id"]}] Q: {row["question"][:100]}')
    print(f'       Author ROUGE: {row["author_rouge"]:.3f} | Ours ROUGE: {row["ours_rouge"]:.3f}')

print('\n--- Ours CORRECT, Author WRONG ---')
for _, row in error_df[(~error_df['author_correct']) & (error_df['ours_correct'])].head(5).iterrows():
    print(f'  [{row["id"]}] Q: {row["question"][:100]}')
    print(f'       Author ROUGE: {row["author_rouge"]:.3f} | Ours ROUGE: {row["ours_rouge"]:.3f}')

In [ ]:
# Refusal details: which retriever's chunks lead to correct refusal?
print('=== Refusal Details (Unanswerable Questions) ===\n')
print(f'{"ID":<4} {"Question":<60} {"Author":<10} {"Ours":<10}')
print('-' * 84)
for a_d, o_d in zip(author_ref_d, ours_ref_d):
    a_status = 'REFUSED' if a_d['refused'] else 'MISSED'
    o_status = 'REFUSED' if o_d['refused'] else 'MISSED'
    print(f'{a_d["id"]:<4} {a_d["question"][:58]:<60} {a_status:<10} {o_status:<10}')

# Show missed refusals
print('\n--- Missed Refusals (Author) ---')
for d in author_ref_d:
    if not d['refused']:
        print(f'  [{d["id"]}] Q: {d["question"][:80]}')
        print(f'       Response: {d["response"][:120]}')

print('\n--- Missed Refusals (Ours) ---')
for d in ours_ref_d:
    if not d['refused']:
        print(f'  [{d["id"]}] Q: {d["question"][:80]}')
        print(f'       Response: {d["response"][:120]}')

## 6. Plots

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart: Author vs Ours
metrics_names = ['Answer Accuracy', 'Refusal Recall']
x = np.arange(len(metrics_names))
w = 0.35
bars1 = axes[0].bar(x - w/2, [author_acc, author_ref], w, label='Author', color='#E5E90C', edgecolor='#2F2F2F')
bars2 = axes[0].bar(x + w/2, [ours_acc, ours_ref], w, label='Ours (CORAL)', color='#FF6B9D', edgecolor='#2F2F2F')
for bar, val in zip(bars1, [author_acc, author_ref]):
    axes[0].text(bar.get_x() + bar.get_width()/2, val + 1.5, f'{val}%', ha='center', fontsize=10)
for bar, val in zip(bars2, [ours_acc, ours_ref]):
    axes[0].text(bar.get_x() + bar.get_width()/2, val + 1.5, f'{val}%', ha='center', fontsize=10)
axes[0].set_xticks(x)
axes[0].set_xticklabels(metrics_names)
axes[0].set_ylim(0, 110)
axes[0].set_ylabel('Score (%)')
axes[0].set_title('Table 2: Author vs Ours Retriever')
axes[0].legend(frameon=False)

# Scatter: ROUGE-L scores Author vs Ours
axes[1].scatter(error_df['author_rouge'], error_df['ours_rouge'],
                alpha=0.6, s=30, c='#FF6B9D', edgecolors='#2F2F2F', linewidths=0.5)
axes[1].plot([0, 1], [0, 1], '--', color='gray', alpha=0.5)
axes[1].axhline(y=ACCURACY_THRESHOLD, color='red', linestyle=':', alpha=0.5)
axes[1].axvline(x=ACCURACY_THRESHOLD, color='red', linestyle=':', alpha=0.5)
axes[1].set_xlabel('Author ROUGE-L')
axes[1].set_ylabel('Ours ROUGE-L')
axes[1].set_title('Per-query Score (above diagonal = Ours better)')

for ax in axes:
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'table2_comparison.png'), dpi=150, bbox_inches='tight')
plt.show()

## 7. Save Results

In [ ]:
results = {
    'summary': {
        'author': {'answer_accuracy': author_acc, 'refusal_recall': author_ref},
        'ours': {'answer_accuracy': ours_acc, 'refusal_recall': ours_ref},
    },
    'author_responses': [
        {'id': item['id'], 'question': item['turns'][0],
         'answerable': item['answerable'], 'reference': item['answer'],
         'response': resp, 'chunks_used': author_chunks[i]}
        for i, (item, resp) in enumerate(zip(test_data, author_responses))
    ],
    'ours_responses': [
        {'id': item['id'], 'question': item['turns'][0],
         'answerable': item['answerable'], 'reference': item['answer'],
         'response': resp, 'chunks_used': ours_chunks[i]}
        for i, (item, resp) in enumerate(zip(test_data, ours_responses))
    ],
}

path = os.path.join(OUTPUT_DIR, 'table2_full_results.json')
with open(path, 'w') as f:
    json.dump(results, f, ensure_ascii=False, indent=2)

error_df.to_csv(os.path.join(OUTPUT_DIR, 'error_analysis.csv'), index=False)

print(f'Saved to {OUTPUT_DIR}/')
print(f'  table2_full_results.json (responses + chunks used)')
print(f'  error_analysis.csv')
print(f'  table2_comparison.png')
print(f'\n=== FINAL ===')
print(summary_df.to_string(index=False))